<a href="https://colab.research.google.com/github/matsunagalab/mdzen/blob/main/colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## MDZen on Colab (最短フロー)

1. **Setup**: 依存導入 + リポジトリ取得 + MCPサーバ起動
2. **Step 1**: ADK Webで Clarification → SimulationBrief 生成
3. **Step 2**: ノートブックで workflow 実行（setup_agent）
4. **Step 3**: py3Dmol 可視化
5. **Step 4**: ZIPダウンロード

---
## Setup: Install Dependencies

**First time only** - This installs AmberTools, OpenMM, and other required packages.

⏱️ Takes ~2-4 minutes. You can continue reading while it runs.

In [ ]:
#@title ▶️ Run Setup (click to expand code)
import sys
import os
import time
import socket
import subprocess

IN_COLAB = 'google.colab' in sys.modules

#==============================================================================
# API Key Configuration
#==============================================================================
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'
#==============================================================================

def load_dotenv():
    for env_path in ['./.env', '../.env', '/content/.env', '/content/mdzen/.env']:
        try:
            with open(env_path) as f:
                for line in f:
                    line = line.strip()
                    if line and not line.startswith('#') and '=' in line:
                        key, value = line.split('=', 1)
                        os.environ[key.strip()] = value.strip().strip('"').strip("'")
            return True
        except FileNotFoundError:
            continue
    return False

load_dotenv()

if IN_COLAB:
    try:
        from google.colab import userdata
        for k in ['ANTHROPIC_API_KEY', 'OPENAI_API_KEY', 'GOOGLE_API_KEY']:
            try:
                v = userdata.get(k)
                if v: os.environ[k] = v
            except: pass
    except: pass

#==============================================================================
# Model Auto-Detection
#==============================================================================
DEFAULT_MODELS = {
    'anthropic': ('anthropic:claude-haiku-4-5-20251001', 'anthropic:claude-haiku-4-5-20251001'),
    'openai': ('openai:gpt-4o-mini', 'openai:gpt-4o-mini'),
    'google': ('google:gemini-2.0-flash', 'google:gemini-2.0-flash'),
}

detected = None
for k, p in [('ANTHROPIC_API_KEY', 'anthropic'), ('OPENAI_API_KEY', 'openai'), ('GOOGLE_API_KEY', 'google')]:
    if os.environ.get(k):
        detected = p
        print(f"✓ {k}")
        break

if detected:
    clarification_model, setup_model = DEFAULT_MODELS[detected]
    os.environ['MDZEN_CLARIFICATION_MODEL'] = clarification_model
    os.environ['MDZEN_SETUP_MODEL'] = setup_model
    model_name = clarification_model.split(':')[1]
    print(f"✓ Default model: {model_name}")
else:
    print("⚠️ No API key found!")
    print("   Set one of: ANTHROPIC_API_KEY, OPENAI_API_KEY, GOOGLE_API_KEY")

#==============================================================================
# MCP サーバー起動ロジック
#==============================================================================
MCP_SERVERS = [
    ("research_server.py", 8001),
    ("structure_server.py", 8002),
    ("genesis_server.py", 8003),
    ("solvation_server.py", 8004),
    ("amber_server.py", 8005),
    ("md_simulation_server.py", 8006),
    ("literature_server.py", 8008),
]

def check_port(port, timeout=2):
    """Check if a port is listening."""
    try:
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.settimeout(timeout)
        result = sock.connect_ex(('localhost', port))
        sock.close()
        return result == 0
    except:
        return False

def kill_old_servers():
    """Kill any existing MCP server processes."""
    for server, _ in MCP_SERVERS:
        subprocess.run(["pkill", "-f", server], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(1)

def start_mcp_servers(python_cmd, server_dir, pythonpath, log_dir=None):
    """Start all MCP servers in HTTP mode.
    
    IMPORTANT: Both stdout and stderr use DEVNULL to prevent pipe buffer blocking.
    If log_dir is provided, stderr is redirected to log files instead.
    """
    procs = []
    for server, port in MCP_SERVERS:
        # Redirect stderr to log file if log_dir provided, otherwise DEVNULL
        if log_dir:
            log_file = open(f"{log_dir}/{server}.log", 'w')
            stderr_target = log_file
        else:
            stderr_target = subprocess.DEVNULL
        
        proc = subprocess.Popen(
            [python_cmd, f"{server_dir}/{server}", "--http", "--port", str(port)],
            stdout=subprocess.DEVNULL,
            stderr=stderr_target,
            env={**os.environ, "PYTHONPATH": pythonpath},
        )
        procs.append((server, port, proc))
    return procs

def wait_for_servers(procs, max_wait=30):
    """Wait for all servers to be ready."""
    print("   Waiting for servers to start...")
    time.sleep(3)
    waited = 3
    while waited < max_wait:
        ready = sum(1 for _, port, proc in procs if check_port(port) and proc.poll() is None)
        if ready == len(procs):
            return True
        time.sleep(1)
        waited += 1
        if waited % 5 == 0:
            print(f"   ... {ready}/{len(procs)} servers ready ({waited}s)")
    return False

def report_server_status(procs, log_dir=None):
    """Report server health status."""
    healthy = 0
    failed = []

    for server, port, proc in procs:
        if proc.poll() is not None:
            # Process exited
            err_msg = ["Process exited unexpectedly"]
            if log_dir:
                try:
                    with open(f"{log_dir}/{server}.log") as f:
                        lines = f.read().strip().split('\n')
                        err_msg = [l for l in lines if 'Error' in l or 'error' in l][-3:] or lines[-3:]
                except:
                    pass
            failed.append((server, port, err_msg))
            print(f"   ✗ {server} (:{port}) - CRASHED")
        elif check_port(port):
            print(f"   ✓ {server} (:{port})")
            healthy += 1
        else:
            err_msg = ["Port not listening"]
            if log_dir:
                try:
                    with open(f"{log_dir}/{server}.log") as f:
                        lines = f.read().strip().split('\n')
                        err_msg = [l for l in lines if 'Error' in l or 'error' in l][-3:] or lines[-3:]
                except:
                    pass
            failed.append((server, port, err_msg))
            print(f"   ? {server} (:{port}) - NOT RESPONDING")

    print(f"✓ {healthy}/{len(procs)} MCP servers running")

    if failed:
        print("\n⚠️ Server startup errors:")
        for server, port, errors in failed:
            print(f"   {server} (:{port}):")
            for err in errors[-3:]:
                print(f"      {err[:120]}")

    return healthy

#==============================================================================
# 環境別セットアップ
#==============================================================================
if IN_COLAB:
    start_time = time.time()
    os.chdir('/content')

    print("📦 Installing Miniforge...")
    !curl -fsSL https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -o /tmp/miniforge.sh
    !bash /tmp/miniforge.sh -b -p /usr/local -u 2>&1 | tail -1
    os.environ["PATH"] = f"/usr/local/bin:{os.environ['PATH']}"
    print(f"✓ Miniforge ({time.time() - start_time:.0f}s)")

    print("⚗️ Installing scientific packages...")
    !mamba install -y -q openmm pdbfixer parmed ambertools rdkit 2>&1 | tail -2
    print(f"✓ Conda packages ({time.time() - start_time:.0f}s)")

    print("📥 Cloning repository...")
    !rm -rf /content/mdzen
    !git clone -q https://github.com/matsunagalab/mdzen.git /content/mdzen
    os.chdir('/content/mdzen')

    print("📦 Installing Python packages (uv)...")
    !pip install -q uv

    CONDA_PYTHON = "/usr/local/bin/python"
    !uv pip install --python {CONDA_PYTHON} -q \
        "litellm>=1.60.0,<1.80.0" \
        anthropic google-genai google-adk \
        "fastmcp>=2.0.0" "mcp[cli]" \
        gradio py3Dmol nest_asyncio \
        mdtraj gemmi pdb2pqr propka dimorphite-dl \
        pubchempy tavily-python

    !uv pip install --python {sys.executable} -q \
        py3Dmol nest_asyncio mdtraj \
        google-adk litellm anthropic \
        "fastmcp>=2.0.0" "mcp[cli]"
    print(f"✓ Pip packages ({time.time() - start_time:.0f}s)")

    os.environ["AMBERHOME"] = "/usr/local"
    sys.path.insert(0, '/content/mdzen/src')
    sys.path.insert(0, '/content/mdzen')

    try:
        import fastmcp
        from google.adk.runners import Runner
        print("✓ Core packages verified")
    except ImportError as e:
        print(f"⚠️ Import: {e}")

    # Create log directory for server logs
    LOG_DIR = "/content/mdzen/logs"
    os.makedirs(LOG_DIR, exist_ok=True)

    print("🚀 Starting MCP servers...")
    kill_old_servers()
    
    PYTHON_CMD = "/usr/local/bin/python"
    SERVER_DIR = "/content/mdzen/servers"
    PYTHONPATH = "/content/mdzen/src"

    mcp_procs = start_mcp_servers(PYTHON_CMD, SERVER_DIR, PYTHONPATH, log_dir=LOG_DIR)
    wait_for_servers(mcp_procs)
    report_server_status(mcp_procs, log_dir=LOG_DIR)

    import logging
    class F(logging.Filter):
        def filter(self, r): return 'cancel scope' not in r.getMessage()
    logging.getLogger('asyncio').addFilter(F())

    print(f"\n✅ Setup complete! ({(time.time() - start_time)/60:.1f} min)")

else:
    sys.path.insert(0, './src')
    load_dotenv()

    LOG_DIR = "./logs"
    os.makedirs(LOG_DIR, exist_ok=True)

    print("🚀 Starting MCP servers (local)...")
    kill_old_servers()
    
    PYTHON_CMD = sys.executable
    SERVER_DIR = "./servers"
    PYTHONPATH = "./src"

    mcp_procs = start_mcp_servers(PYTHON_CMD, SERVER_DIR, PYTHONPATH, log_dir=LOG_DIR)
    wait_for_servers(mcp_procs)
    report_server_status(mcp_procs, log_dir=LOG_DIR)

    print("\n✅ Local setup complete!")

---
## Step 1: Clarification (Google ADK Web)

ここでは **Google ADK Web (`adk web`)** のチャットUIを使って、MDZen に対話的に要件を伝え、**SimulationBrief** を生成します。

- Clarification + Brief生成: **ADK Web 上で実施**
- 実行 (Phase 2 Setup): 次の Step 2 セルで実施
- 可視化: Step 3 (py3Dmol)

まず次のセルで、ADK Web が読み込める形で MDZen の Clarification 用エージェントを生成します。

In [ ]:
#@title 🧩 Step 1a: Generate ADK Web agent package { display-mode: "form" }
#@markdown ADK Web が読み込める形で、Clarification + Brief生成専用の agent package を作ります。

#@markdown ### Output file (bridge between ADK Web → notebook)
state_file = "/content/mdzen/outputs/adk_web_last_run.json" #@param {type:"string"}

import os
import sys
import textwrap
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if not IN_COLAB:
    print("⚠️ This cell is designed for Google Colab. Local Jupyter may need path tweaks.")

# Ensure repo path
repo_dir = Path("/content/mdzen" if IN_COLAB else ".").resolve()
agent_pkg_dir = repo_dir / "mdzen_adk_web"
agent_pkg_dir.mkdir(parents=True, exist_ok=True)

# Make sure adk web process can import mdzen
# (This environment variable will also be used by the later 'adk web' launcher cell.)
os.environ["PYTHONPATH"] = f"{repo_dir}/src:{repo_dir}"

# Bridge path (ADK Web → notebook)
os.environ["MDZEN_ADK_WEB_STATE_FILE"] = state_file

(agent_pkg_dir / "__init__.py").write_text(
    "from .agent import agent\n",
    encoding="utf-8",
)

agent_py = """\
import json
import os
import random
import string
from pathlib import Path

from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.tools import ToolContext
from google.adk.tools.function_tool import FunctionTool

from mdzen.config import get_litellm_model
from mdzen.prompts import get_clarification_instruction
from mdzen.tools.custom_tools import generate_simulation_brief
from mdzen.tools.mcp_setup import get_clarification_tools_sse

DEFAULT_STATE_PATH = os.environ.get(
    "MDZEN_ADK_WEB_STATE_FILE",
    "/content/mdzen/outputs/adk_web_last_run.json",
)


def get_session_dir(tool_context: ToolContext) -> str:
    session_dir = tool_context.state.get("session_dir")
    if session_dir:
        return str(session_dir)

    base_dir = Path("/content/mdzen/outputs")
    base_dir.mkdir(parents=True, exist_ok=True)
    job_id = "".join(random.choices(string.ascii_lowercase + string.digits, k=8))
    session_dir = base_dir / f"job_{job_id}"
    session_dir.mkdir(parents=True, exist_ok=True)

    tool_context.state["session_dir"] = str(session_dir)
    return str(session_dir)


def persist_mdzen_colab_state(
    path: str = DEFAULT_STATE_PATH,
    tool_context: ToolContext = None,
) -> dict:
    state = tool_context.state if tool_context is not None else {}
    payload = {
        "session_dir": state.get("session_dir", ""),
        "simulation_brief": state.get("simulation_brief", {}),
    }
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
    return {"success": True, "path": str(p)}


mcp_tools = get_clarification_tools_sse(host="localhost")

get_session_dir_tool = FunctionTool(get_session_dir)
generate_brief_tool = FunctionTool(generate_simulation_brief)
persist_tool = FunctionTool(persist_mdzen_colab_state)

instruction = (
    get_clarification_instruction()
    + "\n\n"
    + "## ADK Web (Colab) Bridge\n"
    + "After you call generate_simulation_brief and it succeeds, you MUST call persist_mdzen_colab_state() exactly once.\n"
    + f"Write the state file to: {DEFAULT_STATE_PATH}\n"
    + "Then tell the user to return to the notebook and run 'Step 1c' to load it.\n"
)

agent = LlmAgent(
    model=LiteLlm(model=get_litellm_model(\"clarification\")),
    name=\"mdzen_adk_web_clarification\",
    description=\"MDZen Clarification + SimulationBrief generation (for ADK Web in Colab)\",
    instruction=instruction,
    tools=mcp_tools + [get_session_dir_tool, generate_brief_tool, persist_tool],
)
"""

(agent_pkg_dir / "agent.py").write_text(textwrap.dedent(agent_py), encoding="utf-8")

print("=" * 60)
print("✅ ADK Web agent package generated")
print("=" * 60)
print(f"📁 Repo: {repo_dir}")
print(f"📦 Agent package: {agent_pkg_dir}")
print(f"🧾 Bridge state file: {state_file}")
print("\n👉 Next: run Step 1b to start ADK Web UI")


In [ ]:
#@title 🌐 Step 1b: Start Google ADK Web UI { display-mode: "form" }
#@markdown ADK Web のUIを起動して、ブラウザから Clarification + Brief生成を行います。

adk_web_port = 8000 #@param {type:"integer"}

import os
import sys
import time
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

repo_dir = Path("/content/mdzen" if IN_COLAB else ".").resolve()
logs_dir = repo_dir / "logs"
logs_dir.mkdir(parents=True, exist_ok=True)

# Ensure MDZen import path for the adk web subprocess
os.environ["PYTHONPATH"] = f"{repo_dir}/src:{repo_dir}"

# Stop any previous adk web
subprocess.run(["pkill", "-f", "adk web"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

time.sleep(1)

print("🚀 Starting ADK Web...")
print(f"   repo_dir: {repo_dir}")
print(f"   port: {adk_web_port}")

stdout_log = open(logs_dir / "adk_web.stdout.log", "w")
stderr_log = open(logs_dir / "adk_web.stderr.log", "w")

# IMPORTANT: run from the parent directory that contains mdzen_adk_web/
proc = subprocess.Popen(
    ["adk", "web", "--host", "0.0.0.0", "--port", str(adk_web_port)],
    cwd=str(repo_dir),
    stdout=stdout_log,
    stderr=stderr_log,
    env=os.environ.copy(),
)

time.sleep(2)
print("✅ ADK Web started (background)")
print(f"   PID: {proc.pid}")

if IN_COLAB:
    from google.colab import output

    # Opens the ADK Web UI in a new Colab window
    output.serve_kernel_port_as_window(adk_web_port)
    print("\n👉 Opened ADK Web in a new window.")
    print("   In the UI, select the agent: mdzen_adk_web (or mdzen_adk_web_clarification).")
    print("   Chat until SimulationBrief is generated, then it will persist state automatically.")
else:
    print(f"\n👉 Open: http://localhost:{adk_web_port}")
    print("   Select the mdzen_adk_web agent and chat to generate SimulationBrief.")


In [ ]:
#@title 📥 Step 1c: Load SimulationBrief from ADK Web { display-mode: "form" }
#@markdown ADK Web で生成・保存された `SimulationBrief` と `session_dir` を読み込みます。

state_file = "/content/mdzen/outputs/adk_web_last_run.json" #@param {type:"string"}

import json
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

p = Path(state_file)
if not p.exists():
    print("❌ State file not found.")
    print("   Make sure you completed the ADK Web chat and it called persist_mdzen_colab_state.")
    print(f"   Expected: {p}")
else:
    data = json.loads(p.read_text(encoding="utf-8"))
    session_dir = data.get("session_dir")
    brief = data.get("simulation_brief")

    if not session_dir or not brief:
        print("❌ State file exists but is missing required fields.")
        print(f"   session_dir: {bool(session_dir)}")
        print(f"   simulation_brief: {bool(brief)}")
        print("\nRaw file:")
        print(json.dumps(data, indent=2, ensure_ascii=False)[:2000])
    else:
        mdzen_state = {
            "session_dir": session_dir,
            "simulation_brief": brief,
            "workflow_outputs": {},
        }

        print("=" * 60)
        print("✅ Loaded SimulationBrief")
        print("=" * 60)
        print(f"📁 session_dir: {session_dir}")
        print("\n📋 SimulationBrief (summary):")
        # Print a compact summary
        keys = [
            "pdb_id",
            "fasta_sequence",
            "select_chains",
            "include_types",
            "solvation_type",
            "water_model",
            "force_field",
            "temperature",
            "pressure_bar",
            "simulation_time_ns",
        ]
        for k in keys:
            if k in brief:
                print(f"  • {k}: {brief.get(k)}")

        print("\n👉 Next: run Step 2 to execute the workflow")


In [ ]:
#@title ✅ Step 1d: (Optional) Edit brief manually in notebook { display-mode: "form" }
#@markdown ADK Web に戻らずに、簡単な値だけ手動で上書きしたい場合に使います（基本は ADK Web 側で調整がおすすめ）。

#@markdown ### Overrides (leave empty to keep)
_override_temperature = "" #@param {type:"string"}
_override_sim_time_ns = "" #@param {type:"string"}
_override_pressure_bar = "" #@param {type:"string"}

import json

if "mdzen_state" not in dir() or not mdzen_state.get("simulation_brief"):
    print("❌ No SimulationBrief loaded. Run Step 1c first.")
else:
    brief = mdzen_state["simulation_brief"]

    def _to_float(x: str):
        if x is None:
            return None
        x = str(x).strip()
        if x == "":
            return None
        return float(x)

    changed = {}

    try:
        t = _to_float(_override_temperature)
        if t is not None:
            brief["temperature"] = t
            changed["temperature"] = t

        st = _to_float(_override_sim_time_ns)
        if st is not None:
            brief["simulation_time_ns"] = st
            changed["simulation_time_ns"] = st

        p = _to_float(_override_pressure_bar)
        if _override_pressure_bar.strip() == "":
            p = None
        # Allow explicit None via empty string only if user types 'None'
        if str(_override_pressure_bar).strip().lower() == "none":
            p = None
        if _override_pressure_bar.strip() != "":
            brief["pressure_bar"] = p
            changed["pressure_bar"] = p

    except Exception as e:
        print(f"❌ Invalid override: {e}")
        raise

    mdzen_state["simulation_brief"] = brief

    print("✅ Brief ready")
    if changed:
        print("\nUpdated fields:")
        for k, v in changed.items():
            print(f"  • {k}: {v}")

    print("\nCurrent brief (first 40 keys):")
    print(json.dumps({k: brief[k] for k in list(brief.keys())[:40]}, indent=2, ensure_ascii=False))


---
## Step 2: Run MD Workflow

This will execute all 4 steps automatically:
1. **prepare_complex** - Download structure and prepare proteins/ligands
2. **solvate** - Add water box and ions  
3. **build_topology** - Generate Amber topology files
4. **run_simulation** - Run MD with OpenMM

Click ▶️ to start. Progress will be shown below.

In [ ]:
#@title ⚙️ Step 2: Run Complete Workflow { display-mode: "form" }
#@markdown ### Run Options
run_simulation_step = True #@param {type:"boolean"}
#@markdown > Uncheck to skip the MD simulation (for testing setup only)

import json
import time
import traceback
from pathlib import Path

if "mdzen_state" not in dir() or not mdzen_state.get("simulation_brief"):
    print("❌ Error: Run Step 1c first (load SimulationBrief)")
elif "MCP_SERVERS" not in globals() or "check_port" not in globals():
    print("❌ Error: Run the Setup cell first (it defines MCP server helpers)")
else:
    brief = mdzen_state["simulation_brief"]
    session_dir = Path(mdzen_state["session_dir"])
    session_dir.mkdir(parents=True, exist_ok=True)

    # If any MCP server is down, restart all (simple + robust)
    ports = [port for _, port in MCP_SERVERS]
    if not all(check_port(p) for p in ports):
        print("⚠️ Some MCP servers are down. Restarting all...")
        kill_old_servers()
        mcp_procs = start_mcp_servers(PYTHON_CMD, SERVER_DIR, PYTHONPATH, log_dir=LOG_DIR)
        wait_for_servers(mcp_procs)
        report_server_status(mcp_procs, log_dir=LOG_DIR)

    from mdzen import config
    config.settings = config.Settings()

    from mdzen.agents.setup_agent import create_setup_agent
    from mdzen.tools.mcp_setup import close_toolsets

    from google.adk.runners import Runner
    from google.adk.sessions import InMemorySessionService
    from google.genai import types

    async def run_setup():
        agent, mcp_tools = create_setup_agent(transport="http")
        session_service = InMemorySessionService()
        runner = Runner(app_name="mdzen", agent=agent, session_service=session_service)

        initial_state = {
            "session_dir": str(session_dir),
            "simulation_brief": json.dumps(brief) if isinstance(brief, dict) else brief,
            "completed_steps": json.dumps([]),
            "outputs": json.dumps({}),
        }

        session = await session_service.create_session(
            app_name="mdzen",
            user_id="colab_user",
            state=initial_state,
        )

        request = (
            "Execute the MD setup workflow using this SimulationBrief:\n\n"
            + json.dumps(brief, indent=2)
            + "\n\n"
            + f"Work in the directory: {session_dir}\n"
        )
        if not run_simulation_step:
            request += "\nStop after build_topology (do NOT run run_simulation).\n"

        message = types.Content(role="user", parts=[types.Part(text=request)])

        final_text = None
        async for event in runner.run_async(user_id="colab_user", session_id=session.id, new_message=message):
            if event.is_final_response() and event.content and event.content.parts:
                final_text = event.content.parts[0].text

        updated = await session_service.get_session(
            app_name="mdzen", user_id="colab_user", session_id=session.id
        )

        return final_text, updated.state, mcp_tools

    try:
        t0 = time.time()
        final_response, session_state, mcp_tools = await run_setup()
        await close_toolsets(mcp_tools)

        outputs = session_state.get("outputs", {})
        if isinstance(outputs, str):
            try:
                outputs = json.loads(outputs)
            except Exception:
                outputs = {}

        mdzen_state["workflow_outputs"] = outputs

        print("=" * 60)
        print("✅ Workflow finished")
        print("=" * 60)
        print(f"⏱️ Time: {(time.time() - t0)/60:.1f} min")
        print(f"📁 Output: {session_dir}")
        if outputs:
            print("\n📦 Outputs:")
            for k, v in outputs.items():
                if v:
                    print(f"  • {k}: {v}")
        if final_response:
            print("\n📝 Agent summary (truncated):")
            print((final_response[:800] + "...") if len(final_response) > 800 else final_response)

        print("\n👉 Next: Step 3 (visualize)")

    except Exception as e:
        print("❌ Error:", e)
        print(traceback.format_exc())


---
## Step 3: Visualize Results

View the trajectory animation with py3Dmol.

In [ ]:
#@title 🔬 Step 3: Visualize { display-mode: "form" }
#@markdown ### Visualization Options
style = "cartoon" #@param ["cartoon", "stick", "sphere", "line"]
show_water = True #@param {type:"boolean"}

import tempfile
from pathlib import Path

import py3Dmol

if "mdzen_state" not in dir() or not mdzen_state.get("workflow_outputs"):
    print("❌ Error: Please run Step 2 first")
else:
    outputs = mdzen_state["workflow_outputs"]

    # Prefer trajectory if present
    traj_path = outputs.get("trajectory") or outputs.get("trajectory_file")
    top_path = outputs.get("parm7") or outputs.get("prmtop")

    # Fallbacks (if simulation was skipped)
    pdb_path = outputs.get("solvated_pdb") or outputs.get("merged_pdb")

    def _show_from_pdb(pdb_text: str):
        view = py3Dmol.view(width=800, height=500)
        view.addModel(pdb_text, "pdb")
        if style == "cartoon":
            view.setStyle({"cartoon": {"color": "spectrum"}})
        elif style == "stick":
            view.setStyle({"stick": {}})
        elif style == "sphere":
            view.setStyle({"sphere": {"radius": 0.5}})
        else:
            view.setStyle({"line": {}})
        view.zoomTo()
        return view

    if traj_path and top_path:
        print("📊 Loading final trajectory frame...")
        import mdtraj as md

        traj = md.load(traj_path, top=top_path)
        frame = traj[-1]
        frame.image_molecules(inplace=True)

        if not show_water:
            frame = frame.atom_slice(frame.topology.select("protein"))

        with tempfile.NamedTemporaryFile(suffix=".pdb", delete=False, mode="w") as tmp:
            frame.save_pdb(tmp.name, force_overwrite=True)
            tmp_path = Path(tmp.name)

        pdb_text = tmp_path.read_text()
        tmp_path.unlink(missing_ok=True)

        view = _show_from_pdb(pdb_text)
        print("✅ Displaying final structure")
        view.show()

    elif pdb_path:
        p = Path(pdb_path)
        if not p.exists():
            print(f"❌ PDB file not found: {p}")
        else:
            print("📄 No trajectory found. Showing a structure snapshot instead.")
            view = _show_from_pdb(p.read_text())
            view.show()

    else:
        print("❌ Nothing to visualize.")
        print("   Expected either trajectory+parm7 or solvated_pdb/merged_pdb in outputs.")


---
## Step 4: Download Results

Download all generated files as a ZIP archive.

In [ ]:
#@title 📥 Step 4: Download Results { display-mode: "form" }

import sys
from pathlib import Path

if 'mdzen_state' not in dir() or not mdzen_state.get("session_dir"):
    print("❌ Error: Please run the workflow first")
else:
    session_dir = Path(mdzen_state["session_dir"])
    
    if not session_dir.exists():
        print("❌ Error: Session directory not found")
    else:
        print("=" * 50)
        print(f"  📂 {session_dir.name}")
        print("=" * 50)
        
        files = sorted(session_dir.rglob('*'))
        total_size = 0
        
        for f in files:
            if f.is_file():
                size = f.stat().st_size
                total_size += size
                size_str = f"{size/1024:.1f} KB" if size > 1024 else f"{size} B"
                print(f"  {f.relative_to(session_dir):<40} {size_str:>10}")
        
        print("=" * 50)
        print(f"  Total: {total_size/1024/1024:.2f} MB")
        print("=" * 50)
        
        if 'google.colab' in sys.modules:
            from google.colab import files
            import shutil
            
            zip_path = f"/content/{session_dir.name}.zip"
            shutil.make_archive(zip_path.replace('.zip', ''), 'zip', session_dir)
            
            print(f"\n⬇️ Downloading {session_dir.name}.zip...")
            files.download(zip_path)
        else:
            print(f"\n📁 Files are in: {session_dir}")